# Builder and Singleton Patterns in Python

This notebook demonstrates how to refactor procedural code using the **Builder** and **Singleton** design patterns. We start with a complex employee creation function and a configuration manager that repeatedly reads a file. 

### UML Diagram

Below is an ASCII UML‑style diagram showing the relationship between the builder and the product. The builder accumulates state through `with_…` methods and finally calls `build()` to produce an immutable `Employee` object:


```
+------------------+          build()           +---------------+
|  EmployeeBuilder | ------------------------> |    Employee    |
|------------------|                          |---------------|
| - first_name     |                          | first_name     |
| - last_name      |                          | last_name      |
| - email          |                          | email          |
| - department     |                          | department     |
| - position       |                          | position       |
| - salary         |                          | salary         |
| - start_date     |                          | start_date     |
| - manager_id     |                          | manager_id     |
| - phone          |                          | phone          |
| - address        |                          | address        |
| - emergency_contact|                        | emergency_contact|
| - has_parking    |                          | has_parking    |
| - has_laptop     |                          | has_laptop     |
| - has_vpn_access |                          | has_vpn_access |
| - has_admin_rights|                        | has_admin_rights|
| - office_location|                          | office_location|
| - contract_type  |                          | contract_type  |
|------------------|                          +---------------+
| + with_name()    |
| + with_email()   |
| + with_job()     |
| + with_contact() |
| + with_equipment()|
| + with_access()  |
| + with_office()  |
| + with_contract()|
| + build()        |
+------------------+

DeveloperBuilder extends `EmployeeBuilder` and pre‑configures a standard developer role (department='Engineering', position='Senior Developer', typical equipment and access).
```


### Defining the Product

We'll start by defining the `Employee` class using the standard‐library `dataclasses` module. The class serves as a data container and should **not** contain any construction logic. It holds all the fields needed for an employee record.

In [ ]:

# Define the Employee class as a dataclass
from dataclasses import dataclass
from typing import Optional

@dataclass
class Employee:
    """
    Represents an immutable employee record. Once created, the fields should not be modified.
    """
    first_name: str
    last_name: str
    email: str
    department: str
    position: str
    salary: float
    start_date: str
    manager_id: Optional[int] = None
    phone: Optional[str] = None
    address: Optional[str] = None
    emergency_contact: Optional[str] = None
    has_parking: bool = False
    has_laptop: bool = False
    has_vpn_access: bool = False
    has_admin_rights: bool = False
    office_location: Optional[str] = None
    contract_type: str = 'permanent'

    def __post_init__(self):
        """
        Perform basic validation. Validation is kept here because `Employee` is immutable; any issues raised
        must be caught before an instance is created.
        """
        if not self.first_name or not self.last_name:
            raise ValueError('First and last names are required')
        if not self.email or '@' not in self.email:
            raise ValueError('A valid email address is required')
        if self.salary < 0:
            raise ValueError('Salary cannot be negative')


### Implementing the Builder

The `EmployeeBuilder` maintains internal mutable state for each field. Each setter method returns `self` to allow fluent chaining. The final `build()` method validates required fields and returns an immutable `Employee` instance.

In [ ]:

# Implement the EmployeeBuilder
class EmployeeBuilder:
    """
    Fluent builder for the Employee class. Maintains temporary state before producing an Employee.
    """
    def __init__(self):
        # Initialize all fields to None or sensible defaults
        self._first_name: Optional[str] = None
        self._last_name: Optional[str] = None
        self._email: Optional[str] = None
        self._department: Optional[str] = None
        self._position: Optional[str] = None
        self._salary: Optional[float] = None
        self._start_date: Optional[str] = None
        self._manager_id: Optional[int] = None
        self._phone: Optional[str] = None
        self._address: Optional[str] = None
        self._emergency_contact: Optional[str] = None
        self._has_parking: bool = False
        self._has_laptop: bool = False
        self._has_vpn_access: bool = False
        self._has_admin_rights: bool = False
        self._office_location: Optional[str] = None
        self._contract_type: str = 'permanent'

    # Chainable methods to set each group of fields
    def with_name(self, first_name: str, last_name: str):
        """Set the employee's first and last name."""
        self._first_name = first_name
        self._last_name = last_name
        return self

    def with_email(self, email: str):
        """Set the employee's email address."""
        self._email = email
        return self

    def with_job(self, department: str, position: str, salary: float):
        """Set department, position, and salary."""
        self._department = department
        self._position = position
        self._salary = salary
        return self

    def with_start_date(self, start_date: str):
        """Set the employee's start date."""
        self._start_date = start_date
        return self

    def with_manager(self, manager_id: int):
        """Assign a manager by ID."""
        self._manager_id = manager_id
        return self

    def with_contact(self, phone: Optional[str] = None, address: Optional[str] = None, emergency_contact: Optional[str] = None):
        """Set contact information."""
        self._phone = phone
        self._address = address
        self._emergency_contact = emergency_contact
        return self

    def with_equipment(self, laptop: bool = False, parking: bool = False):
        """Configure equipment: laptop and parking."""
        self._has_laptop = laptop
        self._has_parking = parking
        return self

    def with_access(self, vpn: bool = False, admin: bool = False):
        """Set access privileges such as VPN and admin rights."""
        self._has_vpn_access = vpn
        self._has_admin_rights = admin
        return self

    def with_office(self, office_location: str):
        """Set the office location."""
        self._office_location = office_location
        return self

    def with_contract(self, contract_type: str):
        """Set the type of contract (e.g., permanent, internship)."""
        self._contract_type = contract_type
        return self

    def build(self) -> Employee:
        """
        Validate the collected information and return an immutable Employee instance.
        """
        # Validate required fields before building
        if not self._first_name or not self._last_name:
            raise ValueError('First and last names are required')
        if not self._email or '@' not in self._email:
            raise ValueError('A valid email address is required')
        if self._salary is not None and self._salary < 0:
            raise ValueError('Salary cannot be negative')
        # Create and return the Employee dataclass instance
        return Employee(
            first_name=self._first_name,
            last_name=self._last_name,
            email=self._email,
            department=self._department or '',
            position=self._position or '',
            salary=self._salary or 0.0,
            start_date=self._start_date or '',
            manager_id=self._manager_id,
            phone=self._phone,
            address=self._address,
            emergency_contact=self._emergency_contact,
            has_parking=self._has_parking,
            has_laptop=self._has_laptop,
            has_vpn_access=self._has_vpn_access,
            has_admin_rights=self._has_admin_rights,
            office_location=self._office_location,
            contract_type=self._contract_type
        )


### Extending the Builder for Presets

You can derive specialized builders from `EmployeeBuilder` to represent presets. `DeveloperBuilder`, for example, preconfigures the department, position, equipment, and access rights for a typical developer.

In [ ]:

# Specialized builder for developers
class DeveloperBuilder(EmployeeBuilder):
    def __init__(self, first_name: str, last_name: str, email: str):
        super().__init__()
        # Use the fluent interface to set default developer values
        self.with_name(first_name, last_name)
        self.with_email(email)
        # Default job for a developer
        self.with_job('Engineering', 'Senior Developer', 75000.0)
        # Equipment and access suitable for development
        self.with_equipment(laptop=True, parking=False)
        self.with_access(vpn=True, admin=True)
        # Office location and contract type could also be preset
        self.with_office('Paris HQ')
        self.with_contract('permanent')

    def build(self) -> Employee:
        # Optionally add or override validation specific to developers here
        return super().build()


### Demonstrating the Builder

Let's create two employees using the builder. The first will be assembled step by step, and the second will use the preset `DeveloperBuilder`.

In [ ]:

# Create a developer using the fluent interface
employee = (
    EmployeeBuilder()
    .with_name('John', 'Doe')
    .with_email('john.doe@company.com')
    .with_job('Engineering', 'Senior Developer', 75000)
    .with_start_date('2024-01-15')
    .with_contact(phone='+33612345678')
    .with_equipment(laptop=True, parking=False)
    .with_access(vpn=True, admin=True)
    .with_office('Paris HQ')
    .with_contract('permanent')
    .build()
)
print(employee)

# Create an intern with minimal access
intern = (
    EmployeeBuilder()
    .with_name('Jane', 'Smith')
    .with_email('jane.smith@company.com')
    .with_job('Marketing', 'Intern', 15000)
    .with_start_date('2024-02-01')
    .with_manager(42)
    .with_equipment(laptop=True, parking=False)
    .with_access(vpn=False, admin=False)
    .with_office('Paris HQ')
    .with_contract('internship')
    .build()
)
print(intern)

# Use the DeveloperBuilder preset
dev = DeveloperBuilder('Alice', 'Wonder', 'alice.wonder@company.com').build()
print(dev)


## Singleton Pattern

The **Singleton** pattern ensures that a class has only one instance and provides a global access point to that instance. Refactoring Guru explains that a singleton both controls how many instances a class has and gives clients a global point of access【871546148017937†L8-L23】. This is useful when coordinating shared resources, like configuration settings or database connections.

In the original code, each module called `load_config()` independently, causing the configuration file to be read multiple times. This wasteful I/O makes the application slow and can lead to inconsistent state if modules modify their own copies.

### Implementing the ConfigManager Singleton

We'll implement a `ConfigManager` class that lazily loads configuration data the first time it is needed. Subsequent calls return the same instance without re-reading the file. We use a class attribute `_instance` to store the single instance and a `threading.Lock` to make the implementation thread‑safe.

In [ ]:

# Implementing ConfigManager as a Singleton
import json
from threading import Lock

class ConfigManager:
    """
    Singleton configuration manager. Loads configuration once and provides access via get().
    """
    _instance = None
    _lock = Lock()

    def __init__(self, config_file: str = 'config.json'):
        # Prevent multiple initializations
        if hasattr(self, '_initialized') and self._initialized:
            return
        # Read configuration from file
        with open(config_file, 'r') as f:
            self._config = json.load(f)
        self._initialized = True

    @classmethod
    def get_instance(cls, config_file: str = 'config.json') -> 'ConfigManager':
        """Return the singleton instance, creating it if necessary."""
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = cls(config_file)
        return cls._instance

    def get(self, key: str):
        """Retrieve a configuration value using dot notation (e.g., 'database.host')."""
        parts = key.split('.')
        value = self._config
        for part in parts:
            if isinstance(value, dict) and part in value:
                value = value[part]
            else:
                raise KeyError(f"{key} not found in configuration")
        return value


### Demonstrating the Singleton Configuration Manager

To demonstrate the benefits, we'll create a sample `config.json` file in this notebook and access it through the singleton. Notice how the file is only read once even if we call `get_instance()` multiple times.

In [ ]:

# Create a sample configuration file for demonstration purposes
import os
sample_config = {
    'app': {'name': 'PaymentPlatform', 'debug': True},
    'database': {'host': 'localhost', 'port': 5432},
    'email': {'smtp_host': 'smtp.company.com', 'sender': 'no-reply@company.com'},
    'payment': {'api_key': 'sk_test_123456', 'environment': 'sandbox'}
}
with open('config.json', 'w') as f:
    json.dump(sample_config, f)

# First call loads the file
config1 = ConfigManager.get_instance('config.json')
# Subsequent calls reuse the same instance
config2 = ConfigManager.get_instance('config.json')

# Show that both variables reference the same object
print('Same instance:', config1 is config2)

# Access some values
db_host = config1.get('database.host')
debug_mode = config1.get('app.debug')
print('Database host:', db_host)
print('Debug mode:', debug_mode)


## Conclusion

By refactoring the employee creation logic to use the **Builder** pattern, we avoid parameter bloat and make the code more readable and maintainable. The builder provides a fluent interface that clearly expresses which optional features are being set.

Refactoring the configuration loader using the **Singleton** pattern ensures that the configuration file is read only once and shared across the application. The singleton provides a global point of access while controlling instantiation【871546148017937†L8-L23】. These patterns improve both performance and code clarity in real‑world systems.